In [ ]:
from collections import defaultdict
from math import sqrt

State = tuple[int, int, int, int] # 0 represents the blank

def legal_moves(state: State) -> list[tuple[str, State]]:
    """Return all legal blank moves and resulting board states."""
    blank = state.index(0)
    row, col = divmod(blank, 2)
    moves = []
    for name, dr, dc in (("up", -1, 0), ("down", 1, 0),
                         ("left", 0, -1), ("right", 0, 1)):
        nr, nc = row + dr, col + dc
        if 0 <= nr < 2 and 0 <= nc < 2:
            other = nr * 2 + nc
            board = list(state)
            board[blank], board[other] = board[other], board[blank]
            moves.append((name, tuple(board)))
    return moves

def quantum_step(superposition: dict[State, complex]) -> dict[State, complex]:
    next_superposition: defaultdict[State, complex] = defaultdict(complex)
    for state, amplitude in superposition.items():
        successors = legal_moves(state)
        if not successors:
            next_superposition[state] += amplitude
            continue
        move_amplitude = amplitude / sqrt(len(successors))
        for _, successor in successors:
            next_superposition[successor] += move_amplitude
    return dict(next_superposition)

def show_board(state: State) -> str:
    """Format a 2x2 state for display."""
    cells = ["_" if value == 0 else str(value) for value in state]
    return f"{cells[0]} {cells[1]}\n{cells[2]} {cells[3]}"

def run(start: State, steps: int = 3) -> dict[State, complex]:
    """Run the production system for a fixed number of parallel steps."""
    if sorted(start) != [0, 1, 2, 3]:
        raise ValueError("State must contain 0, 1, 2, and 3 exactly once.")
    superposition = {start: 1.0 + 0.0j}
    for _ in range(steps):
        superposition = quantum_step(superposition)
    return superposition

if __name__ == "__main__":
    initial_state: State = (1, 2, 0, 3)
    number_of_steps = 3
    final_states = run(initial_state, number_of_steps)

    print("Initial board:")
    print(show_board(initial_state))
    print(f"\nPossible boards after {number_of_steps} steps:")
    for state, amplitude in sorted(final_states.items()):
        probability = abs(amplitude) ** 2
        print(f"\n{show_board(state)}\namplitude={amplitude:.3f}, probability={probability:.3f}")